# Autoformalization Evaluation

In [1]:
import sys, re, spacy
import gale_shapley_algorithm as gsa
sys.path.insert(0, '/home/flopezp/LogicSim')  
import pandas as pd
from logicsim import utils, metrics
from nltk.metrics import distance

In [2]:
dataset_path = '/home/flopezp/Kurosagol/Ongoing/second_round_experiments/{}/{}_all_samples.csv'
model_list = utils.model_list
ex_data = pd.read_csv(dataset_path.format('FOLIO', model_list[0][0].split('/')[1]))
ex_data = ex_data.drop(columns=['prompt_index', 'sample_index', 'prompt_text'])
ex_data.head()

,generated_text
0,<text>\n ∀x (Performs(x) → (Attends(x) ...
1,<text>\n ∀x (Performs(x) → (Attends(x) ...
2,<text>\n ∀x (Performs(x) → (Attends(x) ...
3,<text>\n ∀x (Performs(x) → (Attends(x) ...
4,<text>\n ∀x (Performs(x) → (Attends(x) ...


In [2]:
# EditDistance + GaleShapley isomorphism creation.

def gs_weights(base, objective):
    """
    Dadas dos listas de predicados/constantes, se obtiene la distancia de edición de los valores de la lista base
    con respecto a los valores de la lista objetivo. Formatea la lista para que se pueda usar con la paqutería de
    GSA.

    base = list
    objective = list
    """
    weights = {elem:[] for elem in base}
    for i in range(len(base)):
        current_base = base[i]
        current_base_distances = []

        for j in range(len(objective)):
            current_obj = objective[j]
            dist = distance.edit_distance(current_base, current_obj) # ELEMENTO A MODIFICAR
            current_base_distances.append((dist, current_obj))
            current_base_distances = sorted(current_base_distances)

        current_base_distances = [_[1] for _ in current_base_distances]
        weights[current_base] = current_base_distances
    return weights

def isomorphism(dataset_value, llm_value):
    """
    Genera un isomorfismo entre los valores del conjunto de datos y la autoformalización del modelo de lenguaje.

    dataset_value = str ; 
    llm_value = str ; 

    return values:
    
    isomorfismo = dict ; Isomorphism based on edit distance and gale-shapley
    mixed_iso = dict ; Isomorphism with modified structure for further processing.
    """
    # We extract predicate and constant info
    ds_preds, ds_consts, _, _ = metrics.extract_info(dataset_value, False)
    llm_preds, llm_consts, _, _ = metrics.extract_info(llm_value, False)

    # We separate predicates based on their arity
    ds_arity = metrics.get_arity_list(ds_preds) 
    llm_arity = metrics.get_arity_list(llm_preds)
    gold_arities = list(set(a[1] for a in ds_arity[0]))
    llm_arities = list(set(a[1] for a in llm_arity[0]))

    # Normalization before obtaining weightsa
    ds_predicate_list, llm_predicate_list = [], []
    ds_constants_list = [elem.lower() for elem in ds_consts]
    llm_constants_list = [elem.lower() for elem in llm_consts]
    
    for elem in gold_arities:
        current_pred = [_[0].lower() if _[1] == elem else None for _ in ds_arity[0]]
        while None in current_pred:
            current_pred.remove(None)
        ds_predicate_list.append(current_pred)

    for elem in llm_arities:
        current_llm_pred = [_[0].lower() if _[1] == elem else None for _ in llm_arity[0]]
        while None in current_llm_pred:
            current_llm_pred.remove(None)
        llm_predicate_list.append(current_llm_pred)

    # ds_predicate_list, llm_predicate_list, ds_constants_list, llm_constants_list -> All values needed and sorted
    # for weight measuring and isomorphism creation.
    
    isomorfismo = []

    # Constant matching
    # IF WE WANT TO MODIFY HOW WE MEASURE SIMILARITY BETWEEN SAME-ARITY VALUES WE HAVE TO MODIFY THE FUNCTION gs_weights(a, b)
    gold_const_weight, llm_const_weight = gs_weights(ds_constants_list, llm_constants_list), gs_weights(llm_constants_list, ds_constants_list)
    constant_iso = gsa.create_matching(gold_const_weight, llm_const_weight).matches
    isomorfismo.append(constant_iso)

    arity = len(ds_predicate_list)
    
    # Gale-Shapley
    for i in range(arity):
        gold_current_arity = ds_predicate_list[i]
        llm_current_arity = llm_predicate_list[i]
        gold_pred_weight, llm_pred_weight = gs_weights(gold_current_arity, llm_current_arity), gs_weights(llm_current_arity, gold_current_arity)
        current_arity_iso = gsa.create_matching(gold_pred_weight, llm_pred_weight).matches
        isomorfismo.append(current_arity_iso)
    
    #print('-'*15, 'Isomorfismo', '-'*15) 
    name_agnostic = []
    for i in range(len(isomorfismo)):
        if i == 0:
            #print(f'Constantes: \n {isomorfismo[i]}') # Solo si queremos ver el isomorfismo antes
            const_name_agnostic = [f'const{j}' for j in range(len(isomorfismo[i]))]
            name_agnostic.append(const_name_agnostic)
        else:
            #print(f'Predicados de aridad {i}: \n {isomorfismo[i]}') # Solo si queremos ver el isomorfismo antes
            pred_name_agnostic = [f'pred{i}a{j}' for j in range(len(isomorfismo[i]))]
            name_agnostic.append(pred_name_agnostic)
    
    #print(name_agnostic)
    mixed_iso = {}
    for i in range(len(isomorfismo)):
        iso_current = isomorfismo[i]
        name_current = name_agnostic[i]

        iso_keys = list(iso_current.keys())
        iso_values = list(iso_current.values())
        for j in range(len(iso_current)):
            mixed_iso[name_current[j]] = [iso_keys[j], iso_values[j]]

    #print(isomorfismo)
    #print('-'*15)
    #print(mixed_iso)

    return isomorfismo, mixed_iso


def name_switch(string, mixed_iso):
    """
        string = str ; El texto a anonimizar, permite ser una serie de premisas.
        mixed_iso = dict ; El isomorfismo mixto obtenido del isomophism(a, b).
    """
    split_value = string.split('\n')
    while '' in split_value:
        split_value.remove('')

    split_value = [elem.lower() for elem in split_value]
    modded = []

    # Me cago EN PUTO CRISTO
    # ¿CÓMO QUE CÚBICO CABRÓN?
    # Tenemos que hacer uso de nuestro buen amigo chat.
    for sentence in split_value:
        modded_sentence = sentence
        for elem in mixed_iso:
            changes = mixed_iso[elem]
            for value in changes:
                explicit_search = re.search(rf'\b{value}\b', modded_sentence)
                if explicit_search != None:
                    modded_sentence = re.sub(value, elem, modded_sentence)
        modded.append(utils.lark_fol_to_p9(modded_sentence, metrics.parser))
    return modded


def semantic_equiavlence(dataset, llm, query):
    """
        Takes a DS-Premise and an LLM-Premise and determines semantic equivalence.

        Queries are optional but recommended. Only case where arbitary queries are used
        is for datasets that don't have tagged values (MALLS/Willow).
    """
    iso, mixed = isomorphism(dataset, llm)
    ds_p_ns = name_switch(dataset, mixed)
    llm_p_ns = name_switch(llm, mixed)

    if query != None:
        query_ns = name_switch(query, mixed)[0]
    else:
        pred1, const1 = False, False
        keys = list(mixed.keys())
        if 'pred1a0' in keys:
            pred1 = True
        if 'const0' in keys:
            const1 = True

        if pred1 and const1:
            query_ns = 'pred1a0(const0)'
        if pred1 and (not const1):
            query_ns = 'pred1a0(x)'

    # print values for further analysis
    #print(f'Dataset Value: \n {dataset}')
    #print('-'*20)
    #print(f'LLM Value: \n {llm}')
    #print('-'*20)
    #print(f'Isomorphism: \n {mixed}')
    
    dataset_infer = utils.prove((query_ns, ds_p_ns))
    llm_infer = utils.prove((query_ns, llm_p_ns))
    if dataset_infer ==  llm_infer:
        return True
    else:
        return False

In [11]:
def logicsim_autoformalization(dataset, model):
    """
    Given a dataset and a model, this function evaluates the autoformalization step.

    dataset = str ; 'FOLIO', 'MALLS', 'Willow'
    model = str ; Nombre explícito del modelo.
    """

    print('='*50)
    print(f'\t Dataset: {dataset}')
    print(f' \t Modelo: {model}')

    llm_dataset_path = f'/home/flopezp/Kurosagol/Ongoing/second_round_experiments/{dataset}/{model}_all_samples.csv'
    df = pd.read_csv(llm_dataset_path)
    df = df.drop(columns=['prompt_index', 'sample_index', 'prompt_text'])

    if dataset == 'FOLIO':
        ref_ds = pd.read_json(r'/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_validation.jsonl', lines = True)
        ref_ds_queries = ref_ds['conclusion-FOL'].to_list()
        ref_ds = ref_ds['premises-FOL'].to_list()

    elif dataset == 'MALLS':
        malls_all = load_dataset('yuan-yang/MALLS-v0', split = 'test')
        ref_ds = list(malls_all['FOL'])

    elif dataset == 'Willow':
        willow_all = load_dataset('iedeveci/WillowNLtoFOL', split = 'test')
        ref_ds = list(willow_all['FOL_expression'])

    # Constants for answer cleaning analysis.
    clean_text = [] # All clear and extrated text is stored here!
    none_regex, avg_ans_length = 0, 0
    answer_len = len(df['generated_text'].to_list())

    for elem in df['generated_text'].to_list():
        if type(elem) != str:
            #print(type(elem), elem)
            none_regex +=1
            clean_text.append(None)
            continue
        avg_ans_length += len(elem)
        text_split = elem.split('<text>')
        regex_extraction = re.search(r'(<text>)[A-z0-9∀∃\n⊕→¬∧ \t()"á,∨]+(<\/text>)', elem) # Regex can be modified
        #regex2_extraction = re.search(r'<text>[\\Śą<>≤A-z0-9:á ∀∧→⊕¬←∨∃↔∈()’\'=≠?.\-,\n"]+<\/text>', elem) 
        
        if regex_extraction != None:
            texto = regex_extraction.group()[6:-7]
            cleaned = re.sub('  ', '', texto)
            clean_text.append(cleaned)
        else:
            none_regex += 1
            clean_text.append(None)

    filtered_ans_len = 0
    for elem in clean_text:
        if elem != None:
            filtered_ans_len += len(elem)

    print(f'Longitud promedio de respuesta: {round(avg_ans_length/answer_len, 4)}')
    print(f'Valores totales: {answer_len}')
    print(f'Valores mal generados: {none_regex}. Porcentaje: {round(none_regex/answer_len, 4)*100}%')
    print(f'Longitud promedio de respuesta filtrada: {round(filtered_ans_len/(answer_len - none_regex), 4)}')

    # Parsing and Cardinality
    llm_parse = False
    ds_parse = False
    cardinal_list = []

    llm_parse_count, dataset_parse_count = 0,0
    parsing_pairs = 0

    for i in range(int(len(clean_text)/5)):
        #ds_value = folio_premises[i]
        ds_value = ref_ds[i]
        nones = 0
        for j in range(5):
            current_index = i*5 + j
            #print('-'*50)
            #print('\t Evaluating instance ', current_index)
            #print('-'*50)
            if clean_text[current_index] == None:
                nones += 1
            else:
                ds_parse = metrics.lark_based_parsability(ds_value, parser = metrics.parser)
                llm_parse = metrics.lark_based_parsability(clean_text[current_index], parser = metrics.parser)
                #print('-'*20, 'Parsing', '-'*20)
                #print(f'DS Parses: {ds_parse}')
                #print(f'LLM Autoform Parses: {llm_parse}')
                if llm_parse:
                    llm_parse_count += 1
                if ds_parse:
                    dataset_parse_count += 1

                if llm_parse and ds_parse:
                    parsing_pairs += 1
                    try:
                        #print('-'*20, 'Cardinality', '-'*20)
                        cardinal_equality = metrics.verify_cardinality(ds_value, clean_text[current_index])
                        #print(f'Cardinality Equality: {cardinal_equality[0]}')
                        #print(f'Constant Errors: {cardinal_equality[1]}')
                        #print(f'Predicate Errors: {cardinal_equality[2]}')
                        if cardinal_equality[0] == True:
                            cardinal_list.append((i, current_index))
                    except:
                        a = 67
                        #print('-'*20, 'Cardinality', '-'*20)
                        #print(f'Cardinality Equality: False')
                        #print(f'DS value: {ds_value}')
                        #print(f'LLM value: {clean_text[current_index]}')

        #print('-'*25)
    
    semantic_equiv = 0
    for elem in cardinal_list:
        ds_index = elem[0]
        llm_index = elem[1]
        ds_value = ref_ds[ds_index]
        llm_value = clean_text[llm_index]
        if dataset == 'FOLIO':
            query = ref_ds_queries[ds_index]
        else:   
            query = False
        sem_eq = semantic_equiavlence(ds_value, llm_value, query)

        if sem_eq == True:
            semantic_equiv += 1

    print('\t \t --- PARSING ---')
    print(f'LLM Parse Rate: {round(llm_parse_count/len(clean_text), 4)*100}%')
    print(f'Dataset Parse Rate: {round(dataset_parse_count/len(clean_text), 4)*100}%')
    print(f'Cantidad de instancias que parsean: {parsing_pairs}')
    print('\t \t --- CARDINALITY ---')
    print(f'Cardinality Equivalence Rate: {round(len(cardinal_list)/len(clean_text), 4)*100}%')
    print(f'Cardinal equal values: {len(cardinal_list)}')
    print('\t\t --- SEMANTIC EQUIVALENCE ---')
    print(f'Semantic Equivalence Rate: {round(semantic_equiv/len(clean_text), 4)*100}%')
    print(f'Semantic Equivalence values: {semantic_equiv}')
    print(f'\t \t ¡Fin!')
    print('='*50)


# ===================================
model_list = utils.model_list
for elem in model_list:
    model_id = elem[0].split('/')[1]
    logicsim_autoformalization('FOLIO', model_id)

	 Dataset: FOLIO
 	 Modelo: Qwen3-4B-FP8
Longitud promedio de respuesta: 5249.9202
Valores totales: 1015
Valores mal generados: 178. Porcentaje: 17.54%
Longitud promedio de respuesta filtrada: 194.963
	 	 --- PARSING ---
LLM Parse Rate: 75.47%
Dataset Parse Rate: 71.72%
Cantidad de instancias que parsean: 659
	 	 --- CARDINALITY ---
Cardinality Equivalence Rate: 5.91%
Cardinal equal values: 60
		 --- SEMANTIC EQUIVALENCE ---
Semantic Equivalence Rate: 5.220000000000001%
Semantic Equivalence values: 53
	 	 ¡Fin!
	 Dataset: FOLIO
 	 Modelo: Qwen3-8B-FP8
Longitud promedio de respuesta: 9368.8857
Valores totales: 1015
Valores mal generados: 29. Porcentaje: 2.86%
Longitud promedio de respuesta filtrada: 30.6633
	 	 --- PARSING ---
LLM Parse Rate: 94.28999999999999%
Dataset Parse Rate: 85.81%
Cantidad de instancias que parsean: 844
	 	 --- CARDINALITY ---
Cardinality Equivalence Rate: 5.220000000000001%
Cardinal equal values: 53
		 --- SEMANTIC EQUIVALENCE ---
Semantic Equivalence Rate: 4.73

ZeroDivisionError: division by zero

In [12]:
ts = pd.read_csv('/home/flopezp/Kurosagol/Ongoing/second_round_experiments/FOLIO/gemma-4-12B-it_all_samples.csv')
ts = ts.drop(columns=['prompt_index', 'sample_index', 'prompt_text'])
ts.head()

,generated_text
0,∀x (Perform(x) ⊕ Attend(x))\n ∀x (Atten...
1,∀x (Perform(x) ⊕ Attend(x))\n ∀x (Perfo...
2,∀x (Perform(x) ⊕ Attend(x)) \n ∀x (Perf...
3,∀x (Perform(x) ⊕ Attend(x)) \n ∀x (Perf...
4,∀x (Perform(x) ⊕ Attend(x))\n ∀x (Perfo...


In [14]:
ts_list = ts['generated_text'].to_list()
for elem in ts_list:
    print(elem)
    print('-'*67)

    ∀x (Perform(x) ⊕ Attend(x))
    ∀x (Attend(x) ⊕ Engaged(x))
    ∀x (Engaged(x) → Student(x))
    ∀x (Inactive(x) ∧ Disinterested(x) → ¬Student(x))
    ∀x (Inactive(x) ∧ Disinterested(x) → ¬Attend(x))
    ∀x (Inactive(x) ∧ Disinterested(x) → ¬Perform(x))
    ∀x (Inactive(x) ∧ Disinterested(x) → ¬Chaperone(x))
    ∀x (Inactive(x) ∧ Disinterested(x) → ¬Engaged(x))
    ∀x (Inactive(x) ∧ Disinterested(x) → ¬Chaperone(x) 1
    ∀x (Inactive(x) ∧ Disinterested(x) → ¬Chaperone(x) 2
    ∀x (Inactive(x) ∧ Disinterested(x) → ¬Chaperone(x) 3
    ∀x (Inactive(x) ∧ Disinterested(x) → ¬Chaperone(x) 4
    ∀x (Inactive(x) ∧ Disinterested(x) → ¬Chaperone(x) 5
    ∀x (Inactive(x) _Disinterested(x) -> ¬Chaperone(x) 6
    ∀x (Inactive(x)_Disinterested(x) -> ¬Chaperone(x) 7
    ∀x (Inactive(x)_Disinterested(x) -> ¬Chaperone(x) 8
    ∀x (Inactive(x)_Disinterested(x) -> ¬Chaperone(x) 9
    ∀x (Inactive(x)_Disinterested(x) -> ¬Chaperone(x) 10
    ∀x (Inactive(x)_Disinterested(x) -> ¬Chaperone(x) 11
    ∀x (